# Stock Data Mining

Importing the necessary libraries

In [2]:
import pandas as pd 
import yfinance as yf
import requests
from yfinance import EquityQuery

In [4]:
# This is for bahrain stocks, but the same works for the nyse (just make sure that the ticker is the same one in yfinance)
STOCKS = ["SALAM_BAH.AE", "SEEF.BH"]

data = yf.download(
    tickers= STOCKS,
    start= "2025-11-01",  #Change the date for the period you want
    end= "2026-11-01",
    interval="1d",    #The daily data interval
    group_by="ticker",
    auto_adjust= True,  
    progress= False,
)


Check if the df is not empty
- (if only the data is only needed as a csv file, then stop here. Otherwise, go through the EDA process)

In [6]:
data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 208 entries, 2025-11-03 to 2026-10-29
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   (SALAM_BAH.AE, Open)    206 non-null    float64
 1   (SALAM_BAH.AE, High)    206 non-null    float64
 2   (SALAM_BAH.AE, Low)     206 non-null    float64
 3   (SALAM_BAH.AE, Close)   206 non-null    float64
 4   (SALAM_BAH.AE, Volume)  206 non-null    float64
 5   (SEEF.BH, Open)         199 non-null    float64
 6   (SEEF.BH, High)         199 non-null    float64
 7   (SEEF.BH, Low)          199 non-null    float64
 8   (SEEF.BH, Close)        199 non-null    float64
 9   (SEEF.BH, Volume)       199 non-null    float64
dtypes: float64(10)
memory usage: 17.9 KB


In [12]:
data.head()

Ticker       SEEF.BH                                        SALAM_BAH.AE  \
Price           Open      High       Low     Close   Volume         Open   
Date                                                                       
2025-11-03  0.109083  0.109083  0.109083  0.109083  50000.0     2.037516   
2025-11-04  0.109083  0.109083  0.109083  0.109083      0.0     2.056209   
2025-11-05  0.109083  0.109083  0.109083  0.109083    416.0     2.084248   
2025-11-06  0.109083  0.109083  0.109083  0.109083   5000.0     2.074901   
2025-11-10  0.109083  0.109083  0.109083  0.109083      0.0     2.009477   

Ticker                                              
Price           High       Low     Close    Volume  
Date                                                
2025-11-03  2.056209  2.037516  2.046862  487290.0  
2025-11-04  2.084248  2.056209  2.084248  193744.0  
2025-11-05  2.084248  2.046862  2.046862  278483.0  
2025-11-06  2.074901  2.009477  2.018823  203821.0  
2025-11-10  2.018823  2.009477  2.018823  192598.0

## Saving it as a csv file:

In [ ]:
# Make sure to change the name of the file if needed
data.to_csv("bahrian_stocks_data_1year.csv")

## Stock Availability

Checking the list of stocks available in Bahrain using Yahoo Finance API:

In [10]:
# Official Bahrain Bourse symbols (as of Nov 2026)
bh_symbols = [
    "ABC", "ABRAAJ", "ALBH", "APMTB", "ARIG", "BANADER", "BASREC", "BEYON",
    "BBK", "BCFC", "BFM", "BISB", "BKIC", "BMB", "BMMI", "BNH", "CINECO",
    "CPARK", "DUTYF", "EBRIT", "ESTERAD", "FAMILY", "GFH", "GHG", "INOVEST",
    "ITHMR", "KHALEEJI", "NASS", "NBB", "NHOTEL", "POLTRY", "SALAM", "SEEF",
    "SILAH", "SOLID", "TAKAFUL", "TRAFCO", "UGH", "UGIC", "ZAINBH"
]

# Appending the .BH suffix used by Yahoo Finance for Bahrain Bourse
yf_tickers = [f"{sym}.BH" for sym in bh_symbols]

# Batch check by downloading 5 days of history
df = yf.download(yf_tickers, period="5d", group_by="ticker", progress= False)

available_tickers = []
for ticker in yf_tickers:
    # Skipping the tickers that werent returned in the yfinance download 
    if ticker not in df.columns.get_level_values(0):
        continue
    # Checking if the ticker has atleast one non-null Close price (meaning there is data)
    had_close_data = df[ticker]["Close"].notna().any()
    if had_close_data:
        available_tickers.append(ticker)
    

print(f"Found {len(available_tickers)} active .BH tickers on Yahoo Finance:")
print(available_tickers)

$TRAFCO.BH: No data found, symbol may be delisted
$BANADER.BH: No data found, symbol may be delisted
$ABRAAJ.BH: No data found, symbol may be delisted
$BFM.BH: No data found, symbol may be delisted
$APMTB.BH: No data found, symbol may be delisted
$NHOTEL.BH: No data found, symbol may be delisted
$BMB.BH: No data found, symbol may be delisted
$FAMILY.BH: No data found, symbol may be delisted
$KHALEEJI.BH: No data found, symbol may be delisted
$TAKAFUL.BH: No data found, symbol may be delisted
$SILAH.BH: No data found, symbol may be delisted
$UGH.BH: No data found, symbol may be delisted
$BEYON.BH: No data found, symbol may be delisted
$POLTRY.BH: No data found, symbol may be delisted
$EBRIT.BH: No data found, symbol may be delisted
$NASS.BH: No data found, symbol may be delisted
$CPARK.BH: No data found, symbol may be delisted

17 Failed downloads:
['TRAFCO.BH', 'BANADER.BH', 'ABRAAJ.BH', 'BFM.BH', 'APMTB.BH', 'NHOTEL.BH', 'BMB.BH', 'FAMILY.BH', 'KHALEEJI.BH', 'TAKAFUL.BH', 'SILAH.BH', 

Found 23 active .BH tickers on Yahoo Finance:
['ABC.BH', 'ALBH.BH', 'ARIG.BH', 'BASREC.BH', 'BBK.BH', 'BCFC.BH', 'BISB.BH', 'BKIC.BH', 'BMMI.BH', 'BNH.BH', 'CINECO.BH', 'DUTYF.BH', 'ESTERAD.BH', 'GFH.BH', 'GHG.BH', 'INOVEST.BH', 'ITHMR.BH', 'NBB.BH', 'SALAM.BH', 'SEEF.BH', 'SOLID.BH', 'UGIC.BH', 'ZAINBH.BH']
